# 05 — ML Filter Diagnostics

This notebook inspects the public-safe ML EV and fill-probability diagnostics exported from the private ledger.

The public sample does not ship model artifacts, model paths, raw feature JSON, wallet identifiers, order IDs, or live execution logic. It keeps only scalar diagnostics that make the execution gate auditable.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "sample"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

pd.set_option("display.max_columns", 80)


In [ ]:
from models.ml_filter import (
    load_public_model_decision_diagnostics,
    load_signal_examples,
    run_walk_forward_demo,
)

executions = pd.read_csv(DATA_DIR / "executions_sample.csv")
examples = load_signal_examples(DATA_DIR / "executions_sample.csv")
train_diag, test_diag = run_walk_forward_demo(examples)
model_diag = load_public_model_decision_diagnostics(DATA_DIR / "executions_sample.csv")

print(f"execution rows: {len(executions):,}")
print(f"signal examples: {len(examples):,}")


## Exported private-ledger decision diagnostics

These fields show the private workflow's ML and fill-probability gate decisions in public-safe scalar form.


In [ ]:
diagnostics_table = pd.DataFrame(
    [
        {"metric": "Rows inspected", "value": model_diag.row_count},
        {"metric": "ML enabled rows", "value": model_diag.ml_enabled_count},
        {"metric": "Rows with ML predicted EV", "value": model_diag.ml_scored_count},
        {"metric": "ML passed rows", "value": model_diag.ml_passed_count},
        {"metric": "ML pass rate", "value": model_diag.ml_pass_rate},
        {"metric": "Avg ML predicted EV", "value": model_diag.avg_ml_predicted_ev},
        {"metric": "Avg ML minimum EV threshold", "value": model_diag.avg_ml_min_ev},
        {"metric": "Rows with fill probability", "value": model_diag.fill_probability_count},
        {"metric": "Fill-probability passed rows", "value": model_diag.fill_prob_passed_count},
        {"metric": "Fill-probability pass rate", "value": model_diag.fill_prob_pass_rate},
        {"metric": "Avg fill probability", "value": model_diag.avg_fill_probability},
        {"metric": "Avg fill-probability threshold", "value": model_diag.avg_fill_prob_min_probability},
    ]
)
diagnostics_table


In [ ]:
reason_tables = []
for label, counts in [
    ("ml_reason", model_diag.ml_reason_counts),
    ("fill_prob_reason", model_diag.fill_prob_reason_counts),
]:
    for reason, count in counts:
        reason_tables.append({"type": label, "reason": reason, "count": count})
pd.DataFrame(reason_tables)


## Walk-forward baseline

The public baseline is not the private production model. It is a transparent learned-threshold baseline that demonstrates chronological validation discipline.


In [ ]:
def diagnostics_to_dict(name, diag):
    return {
        "segment": name,
        "rows": diag.row_count,
        "passed": diag.passed_count,
        "pass_rate": diag.pass_rate,
        "labeled_rows": diag.labeled_count,
        "positive_rate_all": diag.positive_rate_all,
        "positive_rate_passed": diag.positive_rate_passed,
        "positive_rate_rejected": diag.positive_rate_rejected,
    }

pd.DataFrame([
    diagnostics_to_dict("train", train_diag),
    diagnostics_to_dict("test", test_diag),
])


## ML pass vs execution outcomes

Because the private workflow uses ML as an upstream gate, `ml_passed=False` rows should not be read as causal proof. They often did not get the same opportunity to become executed trades. The useful interpretation is operational: the ML EV gate controls which theoretical candidates are allowed to become executable candidates.


In [ ]:
analysis = executions.copy()
analysis["ml_passed_bool"] = analysis["ml_passed"].astype(str).str.lower().isin(["true", "1", "yes"])
analysis["filled_bool"] = analysis["filled"].astype(str).str.lower().isin(["true", "1", "yes"])
summary = (
    analysis.groupby("ml_passed_bool")
    .agg(
        rows=("filled_bool", "size"),
        fill_rate=("filled_bool", "mean"),
        avg_signal_edge=("signal_edge", "mean"),
        avg_spread=("signal_spread", "mean"),
        avg_ml_predicted_ev=("ml_predicted_ev", "mean"),
    )
    .reset_index()
)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(summary["ml_passed_bool"].astype(str), summary["fill_rate"])
ax.set_title("Fill rate by exported ML decision")
ax.set_xlabel("ml_passed")
ax.set_ylabel("Fill rate")
ax.grid(True, axis="y", alpha=0.3)
plt.show()


## Author takeaway

The ML filter was introduced as a post-edge EV gate. In private full backtests and live-ledger replay, the ML EV filter improved the strategy relative to running edge logic without that gate. The selected seven-day public sample cannot fully reproduce that improvement because it does not reconstruct the full parameter history, full trade-level capital path, fees, or account-level path dependency.

The public notebook should therefore be read as a gate-diagnostics notebook, not as proof of a stable ML alpha model.
